In [ ]:
%load_ext autoreload

import os
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import numpy as np

import mcfs
import skewerjax as sjx

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = mcfs.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget
# %matplotlib inline

# Load fake_spectra lines

In [ ]:
base_dir = "/home/STORAGE"
sim_name = "TNG100-1"
snap_num = 33
delta_grid = 10.0
res_kms = 1.0

data = mcfs.load_runs.load_data(base_dir=base_dir, sim_name=sim_name, snap_num=snap_num, delta_grid=delta_grid, res_kms=res_kms, preset="lya_si")

In [ ]:
tau_loaded = data["tau"]
v_kms = data["v_kms"]

print(data["field_keys"]["tau"])
print(tau_loaded.shape)

### postproc tau

In [ ]:
_tau_factor = 1e13
array_tau_factors = np.array([1.0, _tau_factor, None, None], dtype=object) # One entry per line in tau_loaded. Use None for lines you want to discard

keep_mask = np.array([f is not None for f in array_tau_factors], dtype=bool) # Keep only entries with a valid scaling factor
tau_factors_kept = np.array(array_tau_factors[keep_mask], dtype=float) # Convert kept factors to float
tau_scaled = tau_loaded[keep_mask] * tau_factors_kept[:, None, None] # Filter tau_loaded and apply the corresponding scaling

In [ ]:
tau_scaled.shape

# overflux

In [ ]:
line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA") # "SiII-1190", "SiII-1193"

In [ ]:
# tau_scaled shape: (N_lines, N_skewers, N_spectral)
tau_cube = mcfs.overflux_tools.OpticalDepthCube(tau=tau_scaled, line_bundle=line_bundle)

print(f"tau shape: {tau_cube.shape}")
print(f"line IDs: {tau_cube.line_ids}")
print(f"species keys: {tau_cube.species_keys}")
print(f"reference line: {tau_cube.ref_line_id}")
print(f"delta_v array: {tau_cube.delta_v}")
print(f"delta_v dict: {tau_cube.delta_v_dict}")

# Example: use the bundle-defined reference-line shifts automatically
tau_shifted = tau_cube.periodic_shift(x=v_kms, period=np.max(v_kms))

# Build per-line overflux array and the total field
overflux, overflux_total = tau_cube.build_overflux(tau=tau_shifted, ens_axis=0)

# Build all non-empty subset product fields
overflux_subsets, subset_labels = tau_cube.build_subset_fields(overflux=overflux, max_order=None, labels=None,   # defaults to line_bundle line IDs
)

print(f"tau_shifted shape: {tau_shifted.shape}")
print(f"overflux shape: {overflux.shape}")
print(f"overflux_total shape: {overflux_total.shape}")
print(f"overflux_subsets shape: {overflux_subsets.shape}")
print(f"subset_labels: {subset_labels}")

# Plots

In [ ]:
n_plot = 1               # number of random skewers to display
seed = 1234              # for reproducibility
tau_plot = tau_cube.tau  # Use tau_cube.tau if you want the unshifted optical depth, Use tau_shifted if you want the shifted optical depth

# Random skewer selection
rng = np.random.default_rng(seed)
ii_list = np.sort(rng.choice(tau_cube.n_skewers, size=min(n_plot, tau_cube.n_skewers), replace=False))

In [ ]:
# Configuration
xlabel = r"$c\, \ln\left( \frac{\lambda}{\lambda_\dagger}\right)\; \left[\mathrm{km\,/\,s}\right]$"

# Shared metadata / styling
line_ids = tau_cube.line_ids
n_lines = tau_cube.n_lines

cmap = plt.get_cmap("tab10")
colors = {line_id: cmap(i % cmap.N) for i, line_id in enumerate(line_ids)}

line_handles = [
    Line2D([0], [0], color=colors[line_id], lw=3, label=line_id) for line_id in line_ids
]
total_handle = Line2D([0], [0], color="k", lw=3, ls=":", alpha=0.7, label="Total")

# Combined plotting
for ii in ii_list:
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

    tau_tot = np.sum(tau_plot[:, ii, :], axis=0)
    F_lines = np.exp(-tau_plot[:, ii, :])
    F_tot = np.prod(F_lines, axis=0)

    for jj, line_id in enumerate(line_ids):
        color = colors[line_id]
        
        axes[0].plot(v_kms, tau_plot[jj, ii], color=color, lw=1.0, alpha=0.7) # Panel 1: optical depth
        axes[1].plot(v_kms, F_lines[jj], color=color, lw=1.0, alpha=0.7) # Panel 2: transmitted flux per line
        axes[2].plot(v_kms, overflux[jj, ii], color=color, lw=1.0, alpha=0.7) # Panel 3: per-line overflux

    # Total curves
    axes[0].plot(v_kms, tau_tot, color="k", lw=2.0, ls=":", alpha=0.7)
    axes[1].plot(v_kms, F_tot,   color="k", lw=2.0, ls=":", alpha=0.7)
    axes[2].plot(v_kms, overflux_total[ii], color="k", lw=2.0, ls=":", alpha=0.7)
    axes[2].axhline(0.0, color="k", lw=1.0, alpha=0.6)

    # Labels / scales
    axes[0].set_ylabel(r"$\tau$")
    axes[0].set_yscale("log")

    axes[1].set_ylabel(r"$e^{-\tau}$")

    axes[2].set_ylabel(r"$\delta$")
    axes[2].set_xlabel(xlabel)

    # Titles
    axes[0].set_title(f"Skewer index: {ii}")

    # Legend only once, in top panel
    axes[0].legend(handles=line_handles + [total_handle], loc="upper right", frameon=True, fontsize=16)

    plt.tight_layout()
    plt.show()

# P1D

In [ ]:
# Initialize the P1D analyzer on the same spectral grid
P1D = mcfs.P1D.P1DAnalyzer(lambda_or_v=v_kms, optical_depth_cube=tau_cube)

# Total P1D
omega_tot, P1D_tot = P1D.compute_total_P1D(
    overflux_total=overflux_total, axis=-1, subtract_mean=False, drop_zero_mode=True
)
P1D_tot_mean = np.mean(P1D_tot, axis=0)

# Full subset-pair catalog
omega, P1D_catalog, by_order = P1D.compute_subset_P1D_catalog(
    subset_fields=overflux_subsets, subset_labels=subset_labels, axis=-1, subtract_mean=False, drop_zero_mode=True,
)
omega, avg_P1D_catalog = P1D.compute_average_P1D_catalog( # Average over skewers
    P1D_catalog=P1D_catalog, average_axis=0, return_std=False, return_sem=True, symmetrize=True
)

# Some checks
print(f"omega_tot shape: {omega_tot.shape}")
print(f"P1D_tot shape: {P1D_tot.shape}")
print(f"P1D_tot_mean shape: {P1D_tot_mean.shape}")

print(f"number of subset labels: {len(subset_labels)}")
print(f"first few subset labels: {subset_labels[:5]}")

print(f"number of P1D catalog entries: {len(P1D_catalog)}")
print(f"available total orders: {sorted(by_order.keys())}")

example_key = next(iter(avg_P1D_catalog))
print(f"example averaged key: {example_key}")
print(f"example label: {avg_P1D_catalog[example_key]['label']}")
print(f"example mean shape: {avg_P1D_catalog[example_key]['P1D_mean'].shape}")
if 'P1D_sem' in avg_P1D_catalog[example_key]:
    print(f"example sem shape: {avg_P1D_catalog[example_key]['P1D_sem'].shape}")

In [ ]:
linestyles = ["-", "--", ":", "-.", (0, (3, 1, 1, 1)), (0, (5, 1))]
cmap = plt.get_cmap("tab10")
colors = [cmap(i) for i in range(cmap.N)]

def _subset_to_text(subset):
    """
    Convert a subset label tuple into a compact readable string.
    Examples:
        ('HI-LyA',) -> 'HI-LyA'
        ('HI-LyA', 'SiIII-1206') -> 'HI-LyA·SiIII-1206'
    """
    subset = tuple(subset)
    if len(subset) == 1:
        return subset[0]
    return "·".join(subset)

def _canonical_sym_pair(A, B):
    """
    Canonical representative of the symmetric pair (A,B) ~ (B,A).
    """
    return min((A, B), (B, A))

In [ ]:
# Keep only one representative of each symmetric pair: (A,B) ~ (B,A)
unique_terms = {}
for (A, B), entry in avg_P1D_catalog.items():
    canon = _canonical_sym_pair(A, B)
    if canon not in unique_terms:
        unique_terms[canon] = entry

# Sort by total order, then lexicographically by subset labels
sorted_items = sorted(unique_terms.items(), key=lambda item: (item[1]["total_order"], item[0][0], item[0][1]))

orders = sorted({entry["total_order"] for _, entry in sorted_items})
order_to_ls = {order: linestyles[i % len(linestyles)] for i, order in enumerate(orders)}

fig, ax = plt.subplots(figsize=(12, 7))

term_handles = []
term_labels = []

for i, ((A, B), entry) in enumerate(sorted_items):
    mean = np.real(entry["P1D_mean"])
    sem = np.asarray(entry.get("P1D_sem", np.zeros_like(mean)))

    order = entry["total_order"]
    ls = order_to_ls[order]
    color = colors[i % len(colors)]

    A_lab = _subset_to_text(A)
    B_lab = _subset_to_text(B)
    label = f"P[{A_lab}|{B_lab}]"

    ax.plot(omega, mean, lw=2, ls=ls, color=color)
    ax.fill_between(omega, mean - sem, mean + sem, color=color, alpha=0.15)

    term_handles.append(Line2D([0], [0], color=color, lw=2))
    term_labels.append(label)

# total spectrum
ax.plot(omega_tot, np.real(P1D_tot_mean), lw=1.2, ls="-", color="k", label="total")

ax.set_xscale("log")
ax.set_yscale("symlog")
ax.set_xlabel(r"$\omega$")
ax.set_ylabel("P1D")

# Legend for terms
leg1 = ax.legend(term_handles, term_labels, loc="upper right", fontsize=16, ncol=1, frameon=True)
ax.add_artist(leg1)

# Legend for perturbative / total order
order_handles = [Line2D([0], [0], color="k", lw=2, ls=order_to_ls[o], label=rf"$\mathcal{{O}}({o})$") for o in orders]
ax.legend(handles=order_handles, loc="lower left", fontsize=16, frameon=True, ncol=len(order_handles))

plt.tight_layout()
plt.show()

# $\xi1D$

In [ ]:
# Convert P1D -> Xi1D
Xi1D = mcfs.Xi1D.Xi1DAnalyzer(lambda_or_v=v_kms, optical_depth_cube=tau_cube)

lags_tot, Xi1D_tot = Xi1D.compute_total_Xi1D_from_P1D(
    P1D_total=P1D_tot, axis=-1, P1D_zero_mode_dropped=True, center_lags=False,
)
Xi1D_tot_mean = np.mean(Xi1D_tot, axis=0)

lags, Xi1D_catalog, by_order_xi = Xi1D.compute_subset_Xi1D_catalog_from_P1D(
    P1D_catalog=P1D_catalog, axis=-1, P1D_zero_mode_dropped=True,   # because drop_zero_mode=True in P1DAnalyzer
    center_lags=False,            # or True if you prefer centered signed lags
)

lags, avg_Xi1D_catalog = Xi1D.compute_average_Xi1D_catalog(
    Xi1D_catalog=Xi1D_catalog, average_axis=0, return_std=False, return_sem=True, symmetrize=True,
)

# --------------------------------------------------
# Some checks
# --------------------------------------------------
print(f"lags_tot shape: {lags_tot.shape}")
print(f"Xi1D_tot shape: {Xi1D_tot.shape}")
print(f"Xi1D_tot_mean shape: {Xi1D_tot_mean.shape}")

print(f"number of Xi1D catalog entries: {len(Xi1D_catalog)}")
print(f"available total orders in Xi1D catalog: {sorted(by_order_xi.keys())}")

example_key = next(iter(avg_Xi1D_catalog))
print(f"example averaged xi key: {example_key}")
print(f"example xi label: {avg_Xi1D_catalog[example_key]['label']}")
print(f"example xi mean shape: {avg_Xi1D_catalog[example_key]['Xi1D_mean'].shape}")
if 'Xi1D_sem' in avg_Xi1D_catalog[example_key]:
    print(f"example xi sem shape: {avg_Xi1D_catalog[example_key]['Xi1D_sem'].shape}")

In [ ]:
# Keep only one representative of each symmetric pair: (A,B) ~ (B,A)
unique_terms = {}
for (A, B), entry in avg_Xi1D_catalog.items():
    canon = _canonical_sym_pair(A, B)
    if canon not in unique_terms:
        unique_terms[canon] = entry

# Sort by total order, then lexicographically by subset labels
sorted_items = sorted(unique_terms.items(), key=lambda item: (item[1]["total_order"], item[0][0], item[0][1]))

orders = sorted({entry["total_order"] for _, entry in sorted_items})
order_to_ls = {order: linestyles[i % len(linestyles)] for i, order in enumerate(orders)}

fig, ax = plt.subplots(figsize=(12, 7))

term_handles = []
term_labels = []

for i, ((A, B), entry) in enumerate(sorted_items):
    mean = np.real(entry["Xi1D_mean"])
    sem = np.asarray(entry.get("Xi1D_sem", np.zeros_like(mean)))

    order = entry["total_order"]
    ls = order_to_ls[order]
    color = colors[i % len(colors)]

    A_lab = _subset_to_text(A)
    B_lab = _subset_to_text(B)
    label = f"ξ[{A_lab}|{B_lab}]"

    ax.plot(lags, mean, lw=2, ls=ls, color=color)
    ax.fill_between(lags, mean - sem, mean + sem, color=color, alpha=0.15)

    term_handles.append(Line2D([0], [0], color=color, lw=2))
    term_labels.append(label)

# total correlation
ax.plot(lags_tot, np.real(Xi1D_tot_mean), lw=1.2, ls="-", color="k", label="total")

ax.set_xlim([0, np.max(lags)])

ax.set_xlabel(r"$\Delta \left[\mathrm{km\,s^{-1}}\right]$")
ax.set_ylabel(r"$\xi_{\mathrm{1D}}$")

# Legend for terms
leg1 = ax.legend(term_handles, term_labels, loc="upper right", fontsize=16, ncol=2, frameon=True)
ax.add_artist(leg1)

# Legend for perturbative / total order
order_handles = [Line2D([0], [0], color="k", lw=2, ls=order_to_ls[o], label=rf"$\mathcal{{O}}({o})$") for o in orders]
ax.legend(handles=order_handles, loc="lower left", fontsize=16, frameon=True, ncol=len(order_handles))

plt.tight_layout()
plt.show()